# 07 · Clean the medical and emergency facility lists

**Replaces** the medical/emergency half of the old `build_trimmed_csvs.py`.

**Reads** (`data/external/services/trimmed/`):

| file | origin |
|---|---|
| `medical_facilities_trimmed.csv` | Geoscience Australia / NHSD `medical_facilities_NT.csv`, already cut to 8 columns |
| `emergency_facilities_trimmed.csv` | GA `emergency_facilities_NORTHERN_TERRITORY.csv`, already cut to 8 columns |

The original raw extracts are not in the repo, so these trimmed snapshots are the input layer.
Their coordinates are supplied by the source (site-level precision).

**Writes** `data/processed/medical_services.csv` and `data/processed/emergency_services.csv`.

**Fixes applied here**

1. `emergency_class` came out as `Ses Facility` (from `str.title()`). It's now `SES Facility`.
2. The AFP office's agency was the raw value `WWW.AFP.GOV.AU` because the old agency map had a key that never matched. It's now `Australian Federal Police`.
3. Ambulance stations were credited to NT Police, Fire and Emergency Services. The NT ambulance service is run by St John Ambulance NT.
4. Emergency sites named only after their town (`DARWIN`, `KATHERINE`, ...) are ambiguous, because one town can have fire, SES and ambulance sites. The facility type is now appended to the name, plus the suburb if the name is still shared.
5. Two `HUMPTY DOO` metro fire facilities about 190 m apart are one site recorded twice, so we drop duplicates (same name and class, within 0.5 km).
6. New flag columns `*_suburb_far_from_site` mark rows whose suburb label is more than 15 km from the other sites with that label. Example: *Ngukurr Health Centre* is labelled `Katherine` but sits 250 km east. nb 08 uses the flag to keep those rows out of the suburb centroids used to geocode schools.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Walk up from wherever Jupyter was started until we find the repo root
# (the folder that holds data/raw), so the notebook runs from any cwd.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
INPUTS = ROOT / "data" / "external" / "services" / "trimmed"   # read-only inputs
PROCESSED = ROOT / "data" / "processed"                         # the only folder we write to

# NT bounding box used as a sanity check on every coordinate we write.
NT_LAT, NT_LON = (-26.1, -10.9), (128.9, 138.1)


def in_nt(df: pd.DataFrame) -> pd.Series:
    return df["latitude"].between(*NT_LAT) & df["longitude"].between(*NT_LON)


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (vectorised; any argument may be an array)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))


## 1. Load the inputs
Postcodes are read as strings so the leading `0` of NT postcodes (`0870`) survives.

In [2]:
med = pd.read_csv(INPUTS / "medical_facilities_trimmed.csv", dtype={"medical_postcode": str})
emg = pd.read_csv(INPUTS / "emergency_facilities_trimmed.csv", dtype={"emergency_postcode": str})

# Basic integrity checks: every row has a coordinate inside the NT box.
for label, df in [("medical", med), ("emergency", emg)]:
    assert df[["latitude", "longitude"]].notna().all().all(), f"{label}: missing coordinates"
    assert in_nt(df).all(), f"{label}: coordinate outside NT"
    print(f"{label}: {len(df)} rows, all coordinates inside the NT")


medical: 181 rows, all coordinates inside the NT
emergency: 152 rows, all coordinates inside the NT


## 2. Flag suburb labels that don't match the site's location

For each suburb we take the **median** point of its sites, which one far-off site can't drag
the way it drags a mean. Any site more than `SUBURB_SPREAD_KM` from that median gets flagged.
Usually the source gave the organisation's postal suburb, not the clinic's own.
We **don't** overwrite the suburb, because it is what the source says. We only mark it.

In [3]:
SUBURB_SPREAD_KM = 15


def flag_far_from_suburb(df: pd.DataFrame, suburb_col: str) -> pd.Series:
    """True where a row sits > SUBURB_SPREAD_KM from the median point of its suburb."""
    key = df[suburb_col].str.upper().str.strip()
    med_lat = df.groupby(key)["latitude"].transform("median")
    med_lon = df.groupby(key)["longitude"].transform("median")
    return pd.Series(haversine_km(df["latitude"], df["longitude"], med_lat, med_lon) > SUBURB_SPREAD_KM,
                     index=df.index)


med["medical_suburb_far_from_site"] = flag_far_from_suburb(med, "medical_suburb")
emg["emergency_suburb_far_from_site"] = flag_far_from_suburb(emg, "emergency_suburb")

print("medical rows flagged:")
print(med.loc[med["medical_suburb_far_from_site"], ["name", "medical_suburb", "latitude", "longitude"]].to_string())
print("\nemergency rows flagged:")
print(emg.loc[emg["emergency_suburb_far_from_site"], ["name", "emergency_suburb", "latitude", "longitude"]].to_string())


medical rows flagged:
                              name medical_suburb   latitude   longitude
7    Sunrise Health Barunga Clinic      Katherine -14.522405  132.864990
10           Ngukurr Health Centre      Katherine -14.733854  134.732086
19          Wugularr Health Centre      Katherine -14.554990  133.113953
32       Manyallaluk Health Centre      Katherine -14.522353  132.864960
122         Minyerri Health Centre      Katherine -15.223181  134.081772

emergency rows flagged:
Empty DataFrame
Columns: [name, emergency_suburb, latitude, longitude]
Index: []


## 3. Medical: review repeated names

Repeated names in the medical list are **legitimate**. NHSD lists one clinic once per service
(e.g. a GP record and a pharmacy record at the same health centre), so we keep them all.

In [4]:
med["name"] = med["name"].str.strip()
dups = med[med["name"].duplicated(keep=False)].sort_values("name")
print(dups[["name", "medical_class", "medical_service_type", "latitude", "longitude"]].to_string())


                                                name medical_class                                  medical_service_type   latitude   longitude
29                          Gunyangara Health Centre            GP             Maternal, child and family health service -12.220892  136.703512
132                         Gunyangara Health Centre      Pharmacy                                      Pharmacy service -12.220892  136.703512
50                    Imanpa Community Health Centre            GP             Maternal, child and family health service -25.127584  132.570557
175                   Imanpa Community Health Centre      Hospital                          Emergency department service -25.127584  132.570557
22                  Kalkaringi Primary Health Centre            GP             Maternal, child and family health service -17.445427  130.831085
131                 Kalkaringi Primary Health Centre      Pharmacy                                      Pharmacy service -17.445427  130

## 4. Emergency: fix class labels and agencies

In [5]:
emg["name"] = emg["name"].str.strip()

# (1) str.title() in the trimming step turned the acronym SES into "Ses".
emg["emergency_class"] = emg["emergency_class"].replace({"Ses Facility": "SES Facility"})

# (2) The AFP row kept its raw FACILITY_SOURCE value because the old map's key never matched.
emg["emergency_agency"] = emg["emergency_agency"].replace({"WWW.AFP.GOV.AU": "Australian Federal Police"})

# (3) Ambulance services in the NT are delivered by St John Ambulance NT, not NTPFES.
emg.loc[emg["emergency_class"] == "Ambulance Station", "emergency_agency"] = "St John Ambulance NT"

print(emg["emergency_class"].value_counts().to_string(), "\n")
print(emg["emergency_agency"].value_counts().to_string())


emergency_class
Policing Facility                      66
SES Facility                           32
Metro Fire Facility                    27
Rural/Country Fire Service Facility    22
Ambulance Station                       5 

emergency_agency
NT Police, Fire and Emergency Services    92
NT Fire and Rescue                        54
St John Ambulance NT                       5
Australian Federal Police                  1


## 5. Emergency: drop duplicate sites

A duplicate is the same name and class within `DUP_KM` of an earlier row. Genuinely distinct
sites in one town have different classes, so they are never merged.

In [6]:
DUP_KM = 0.5

drop_idx = []
for _, grp in emg.groupby(["name", "emergency_class"]):
    kept = []
    for idx, row in grp.iterrows():
        if any(haversine_km(row.latitude, row.longitude, emg.at[k, "latitude"], emg.at[k, "longitude"]) <= DUP_KM
               for k in kept):
            drop_idx.append(idx)          # a near-identical row was already kept
        else:
            kept.append(idx)

print("dropping duplicate emergency rows:")
print(emg.loc[drop_idx, ["name", "emergency_class", "latitude", "longitude"]].to_string())
emg = emg.drop(index=drop_idx).reset_index(drop=True)


dropping duplicate emergency rows:
          name      emergency_class   latitude   longitude
33  HUMPTY DOO  Metro Fire Facility -12.572673  131.100336


## 6. Emergency: make town-only names unambiguous

Rows whose name contains no facility word are fire, SES or ambulance sites named after their town.
We append a short facility label so every map tooltip and "nearest site" name makes sense by itself.
If two different sites still share a name after that, we also add the suburb.

In [7]:
FACILITY_WORDS = r"POLICE|STATION|FIRE|BRIGADE|SES|RESCUE|AMBULANCE|EMERGENCY|OFFICE|CENTRE|UNIT|POST"
CLASS_SUFFIX = {
    "Metro Fire Facility": "FIRE STATION",
    "Rural/Country Fire Service Facility": "FIRE STATION",
    "SES Facility": "SES UNIT",
    "Ambulance Station": "AMBULANCE STATION",
    "Policing Facility": "POLICE FACILITY",
}

town_only = ~emg["name"].str.upper().str.contains(FACILITY_WORDS, regex=True)
emg.loc[town_only, "name"] = emg.loc[town_only, "name"] + " " + emg.loc[town_only, "emergency_class"].map(CLASS_SUFFIX)
print(f"renamed {int(town_only.sum())} town-only rows, e.g.:")
print(emg.loc[town_only, ["name", "emergency_class"]].head(10).to_string())


# Two genuinely different stations can still share a name (e.g. Alice Springs has fire
# stations in Ross and The Gap); add the suburb to tell them apart.
still_dup = emg["name"].duplicated(keep=False)
emg.loc[still_dup, "name"] = emg.loc[still_dup, "name"] + " (" + emg.loc[still_dup, "emergency_suburb"] + ")"
print("\nadded suburb to:", emg.loc[still_dup, "name"].tolist())
assert emg["name"].is_unique


renamed 69 town-only rows, e.g.:
                           name      emergency_class
2   ADELAIDE RIVER FIRE STATION  Metro Fire Facility
6          HARTS RANGE SES UNIT         SES Facility
8        BATCHELOR FIRE STATION  Metro Fire Facility
10      BEES CREEK FIRE STATION  Metro Fire Facility
14      BORROLOOLA FIRE STATION  Metro Fire Facility
15     DARWIN AMBULANCE STATION    Ambulance Station
17       ALICE SPRINGS SES UNIT         SES Facility
27          ELLIOT FIRE STATION  Metro Fire Facility
28             KULGERA SES UNIT         SES Facility
30      KOOLPINYAH FIRE STATION  Metro Fire Facility

added suburb to: ['ALICE SPRINGS FIRE STATION (Ross)', 'ALICE SPRINGS FIRE STATION (The Gap)']


## 7. Write outputs

In [8]:
assert in_nt(med).all() and in_nt(emg).all()
med.to_csv(PROCESSED / "medical_services.csv", index=False)
emg.to_csv(PROCESSED / "emergency_services.csv", index=False)
print(f"wrote medical_services.csv   ({len(med)} rows)")
print(f"wrote emergency_services.csv ({len(emg)} rows)")


wrote medical_services.csv   (181 rows)
wrote emergency_services.csv (151 rows)
